In [ ]:
from pathlib import Path
import joblib
import geopandas as gpd
import pandas as pd
import numpy as np

# ==============================================================================
# 1. Define Pipeline Paths
# ==============================================================================
MODEL_DIR = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Output\Models")
INPUT_GRID_PATH = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\hexagon_ktm.geojson")
OUTPUT_GRID_PATH = Path(r"C:\Users\anils\Desktop\7th SEM\GIS\GeoOptim\Data\Overture\hexagon_ktm_predictions.geojson")

# ==============================================================================
# 2. Load Trained Artifacts and Dataset
# ==============================================================================
print("⏳ Loading model pipeline artifacts...")
scaler = joblib.load(MODEL_DIR / "robust_scaler.pkl")
model_rf = joblib.load(MODEL_DIR / "mlp_multilabel_model.pkl")

print("🗺️ Loading inference geographic grid dataset...")
hex_gdf = gpd.read_file(INPUT_GRID_PATH)

# ==============================================================================
# 3. Dynamic Feature Column Extraction
# ==============================================================================
# Define the non-feature tracking attributes to isolate your dynamic feature matrix
metadata_cols = ['h3_id', 'geometry']
feature_cols = [col for col in hex_gdf.columns if col not in metadata_cols and not col.startswith('prob_')]

# The target multi-label classes your network yields output arrays for
clean_label_names = [
    'Restaurant', 'Hotel_Lodging', 'Malls_Department', 'Cafe_Bakery' 
    , 'Fashion_Clothing', 'Convenience_Specialty'
]

X_inference = hex_gdf[feature_cols].copy()

# Ensure feature counts align with what the scaler expects
expected_features = scaler.n_features_in_
actual_features = len(feature_cols)
if actual_features != expected_features:
    raise ValueError(f"❌ Feature count mismatch! Model expects {expected_features}, but dataset has {actual_features}.")

# ==============================================================================
# 4. Run Scaling and Predict Probabilities (FIXED FOR MLP)
# ==============================================================================
print("⚖️ Normalizing features using loaded RobustScaler...")
X_inference_scaled = scaler.transform(X_inference)

print("🧠 Running Multi-Label MLP model inference engine...")
# ✨ FIX: For MLP Multi-label, y_prob is a single 2D array of shape (n_samples, n_classes)
y_prob_matrix = model_rf.predict_proba(X_inference_scaled)

print("📈 Appending class probabilities to dataset...")
for idx, label_name in enumerate(clean_label_names):
    # Slice the matrix column directly for each target class
    hex_gdf[f'prob_{label_name}'] = y_prob_matrix[:, idx].astype('float32')

# ==============================================================================
# 5. Export Updated Geographic Layer File
# ==============================================================================
print(f"💾 Saving complete prediction probability field matrix...")
hex_gdf.to_file(OUTPUT_GRID_PATH, driver="GeoJSON")

print("\n🎉 Model Inference Run Complete!")
print(f"Saved completed predictions with {len(clean_label_names)} probability fields to: {OUTPUT_GRID_PATH}")